In [1]:
import os
import json
import time
import requests
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem
from matchms.importing import load_from_mgf

RDLogger.DisableLog("rdApp.*")

# =========================
# CONFIG
# =========================
MGF_FILE = r"C:/Users/bibhushaojha/Desktop/RA work/ALL_GNPS.mgf"  
INDEX_JSON_PATH = "C:/Users/bibhushaojha/Desktop/NP-Classifier/Classifier/dict/index_v1.json"

PARSED_CSV = "parsed_spectra_full.csv"
UNIQUE_SMILES_CSV = "unique_smiles_full.csv"
SMILES_LABELS_CSV = "smiles_labels_full.csv"
CACHE_JSON = "smiles_label_cache_full.json"
LABELED_CSV = "spectra_with_labels_full.csv"

TF_SERVING_URL = "http://localhost:8501/v1/models"
CLASS_URL = f"{TF_SERVING_URL}/CLASS:predict"
SUPERCLASS_URL = f"{TF_SERVING_URL}/SUPERCLASS:predict"
PATHWAY_URL = f"{TF_SERVING_URL}/PATHWAY:predict"

PARSE_SAVE_EVERY = 10000
LABEL_SAVE_EVERY = 1000
CHUNK_SIZE = 50000
LOG_EVERY = 1000

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

In [2]:
def get_meta(spec, key, default=None):
    try:
        val = spec.metadata.get(key, default)
        return default if val is None else val
    except Exception:
        return default

def safe_json_dumps(x):
    try:
        return json.dumps(x)
    except Exception:
        return None

def canonicalize_smiles(smiles):
    if smiles is None:
        return None
    smiles = str(smiles).strip()
    if smiles == "":
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

def inchi_to_smiles(inchi):
    if inchi is None:
        return None
    inchi = str(inchi).strip()
    if not inchi.startswith("InChI="):
        return None
    try:
        mol = Chem.MolFromInchi(inchi)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

def extract_canonical_smiles(spec):
    smiles = get_meta(spec, "smiles")
    inchi = get_meta(spec, "inchi")

    smi = canonicalize_smiles(smiles)
    if smi is not None:
        return smi

    smi2 = inchi_to_smiles(inchi)
    if smi2 is not None:
        return canonicalize_smiles(smi2)

    return None

def load_cache(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            cache = json.load(f)
        log(f"Loaded cache with {len(cache):,} entries")
        return cache
    return {}

def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f)
    log(f"Saved cache -> {path}")

In [3]:
log("Testing MGF loading on a tiny sample...")

sample_rows = []
for i, spec in enumerate(load_from_mgf(MGF_FILE), start=1):
    mz = getattr(spec, "mz", None)
    intensities = getattr(spec, "intensities", None)

    mz_list = mz.tolist() if mz is not None else None
    intensity_list = intensities.tolist() if intensities is not None else None

    sample_rows.append({
        "spec_index": i,
        "scan": get_meta(spec, "scans"),
        "compound_name": get_meta(spec, "compound_name"),
        "precursor_mz": get_meta(spec, "precursor_mz"),
        "smiles": extract_canonical_smiles(spec),
        "raw_smiles": get_meta(spec, "smiles"),
        "inchi": get_meta(spec, "inchi"),
        "num_peaks": len(mz_list) if mz_list is not None else 0,
        "mz_json": safe_json_dumps(mz_list),
        "intensity_json": safe_json_dumps(intensity_list)
    })

    if i >= 5:
        break

sample_df = pd.DataFrame(sample_rows)
print(sample_df.shape)
display(sample_df)
print(sample_df.dtypes)

[20:07:01] Testing MGF loading on a tiny sample...
(5, 10)


,spec_index,scan,compound_name,precursor_mz,smiles,raw_smiles,inchi,num_peaks,mz_json,intensity_json
0,1,1,3-Des-Microcystein_LR M+H,981.540,C=C1C(=O)NC(C)C(=O)NC(CC(C)C)C(=O)NC(C(=O)O)C(...,CC(C)CC1NC(=O)C(C)NC(=O)C(=C)N(C)C(=O)CCC(NC(=...,None,218,"[289.286377, 295.545288, 298.489624, 317.32495...","[8068.0, 22507.0, 3925.0, 18742.0, 8604.0, 804..."
1,2,1,Hoiamide B M+H,940.250,CCC[C@@H](C)[C@H](O)[C@H](C)[C@H]1OC(=O)[C@H](...,CCC[C@@H](C)[C@@H]([C@H](C)[C@@H]1[C@H]([C@H](...,InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...,335,"[278.049927, 278.957642, 281.258667, 291.99609...","[35793.0, 47593.0, 95495.0, 115278.0, 91752.0,..."
2,3,1,Malyngamide C M+H,456.100,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)[C@]12O[...,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)/[C@@]12...,InChI=1S/C24H38ClNO5/c1-3-4-5-6-8-11-19(30-2)1...,157,"[128.838745, 132.075684, 132.830322, 134.03843...","[8064.0, 8080.0, 18139.0, 11099.0, 4449.0, 256..."
3,4,1,Scytonemin M+H,545.000,O=C1C(C2=C3C(=Nc4ccccc43)/C(=C\c3ccc(O)cc3)C2=...,OC1=CC=C(\C=C2\C(=O)C(C3=C4C5=C(C=CC=C5)N=C4\C...,InChI=1S/C36H20N2O4/c39-21-13-9-19(10-14-21)17...,40,"[343.896484, 345.458496, 372.684021, 386.09747...","[142503.0, 67939.0, 175247.0, 65864.0, 104575...."
4,5,1,Salinisporamide A M+H,314.116,None,None,None,212,"[101.015465, 101.059807, 103.002441, 103.02854...","[34.201859, 14.90337, 17.01816, 58.15749, 46.2..."


spec_index          int64
scan               object
compound_name      object
precursor_mz      float64
smiles             object
raw_smiles         object
inchi              object
num_peaks           int64
mz_json            object
intensity_json     object
dtype: object


In [4]:
def parse_full_mgf_to_csv():
    if os.path.exists(PARSED_CSV):
        log(f"{PARSED_CSV} already exists. Skipping parse.")
        return

    log("Starting full MGF parsing...")
    start_time = time.time()
    rows = []
    total_saved = 0

    for i, spec in enumerate(load_from_mgf(MGF_FILE), start=1):
        if i % LOG_EVERY == 0:
            elapsed = time.time() - start_time
            log(f"Parsed {i:,} spectra | elapsed {elapsed:.2f} sec")

        mz = getattr(spec, "mz", None)
        intensities = getattr(spec, "intensities", None)

        mz_list = mz.tolist() if mz is not None else None
        intensity_list = intensities.tolist() if intensities is not None else None

        rows.append({
            "spec_index": i,
            "scan": get_meta(spec, "scans"),
            "compound_name": get_meta(spec, "compound_name"),
            "precursor_mz": get_meta(spec, "precursor_mz"),
            "smiles": extract_canonical_smiles(spec),
            "raw_smiles": get_meta(spec, "smiles"),
            "inchi": get_meta(spec, "inchi"),
            "num_peaks": len(mz_list) if mz_list is not None else 0,
            "mz_json": safe_json_dumps(mz_list),
            "intensity_json": safe_json_dumps(intensity_list)
        })

        if len(rows) >= PARSE_SAVE_EVERY:
            chunk_df = pd.DataFrame(rows)
            write_header = not os.path.exists(PARSED_CSV)
            chunk_df.to_csv(PARSED_CSV, mode="a", header=write_header, index=False)
            total_saved += len(rows)
            log(f"Saved chunk | total saved: {total_saved:,}")
            rows = []

    if rows:
        chunk_df = pd.DataFrame(rows)
        write_header = not os.path.exists(PARSED_CSV)
        chunk_df.to_csv(PARSED_CSV, mode="a", header=write_header, index=False)
        total_saved += len(rows)

    log(f"Finished parsing. Total saved: {total_saved:,}")

In [5]:
parse_full_mgf_to_csv()

[20:08:07] parsed_spectra_full.csv already exists. Skipping parse.


In [6]:
log("Checking parsed CSV...")

parsed_head = pd.read_csv(PARSED_CSV, nrows=10)
print("Shape of head sample:", parsed_head.shape)
display(parsed_head)

print("\nColumns:")
print(parsed_head.columns.tolist())

print("\nNull counts in sample:")
print(parsed_head.isnull().sum())

print("\nSample dtypes:")
print(parsed_head.dtypes)

[20:08:16] Checking parsed CSV...
Shape of head sample: (10, 10)


,spec_index,scan,compound_name,precursor_mz,smiles,raw_smiles,inchi,num_peaks,mz_json,intensity_json
0,1,1,3-Des-Microcystein_LR M+H,981.540,C=C1C(=O)NC(C)C(=O)NC(CC(C)C)C(=O)NC(C(=O)O)C(...,CC(C)CC1NC(=O)C(C)NC(=O)C(=C)N(C)C(=O)CCC(NC(=...,NaN,218,"[289.286377, 295.545288, 298.489624, 317.32495...","[8068.0, 22507.0, 3925.0, 18742.0, 8604.0, 804..."
1,2,1,Hoiamide B M+H,940.250,CCC[C@@H](C)[C@H](O)[C@H](C)[C@H]1OC(=O)[C@H](...,CCC[C@@H](C)[C@@H]([C@H](C)[C@@H]1[C@H]([C@H](...,InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...,335,"[278.049927, 278.957642, 281.258667, 291.99609...","[35793.0, 47593.0, 95495.0, 115278.0, 91752.0,..."
2,3,1,Malyngamide C M+H,456.100,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)[C@]12O[...,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)/[C@@]12...,InChI=1S/C24H38ClNO5/c1-3-4-5-6-8-11-19(30-2)1...,157,"[128.838745, 132.075684, 132.830322, 134.03843...","[8064.0, 8080.0, 18139.0, 11099.0, 4449.0, 256..."
3,4,1,Scytonemin M+H,545.000,O=C1C(C2=C3C(=Nc4ccccc43)/C(=C\c3ccc(O)cc3)C2=...,OC1=CC=C(\C=C2\C(=O)C(C3=C4C5=C(C=CC=C5)N=C4\C...,InChI=1S/C36H20N2O4/c39-21-13-9-19(10-14-21)17...,40,"[343.896484, 345.458496, 372.684021, 386.09747...","[142503.0, 67939.0, 175247.0, 65864.0, 104575...."
4,5,1,Salinisporamide A M+H,314.116,NaN,NaN,NaN,212,"[101.015465, 101.059807, 103.002441, 103.02854...","[34.201859, 14.90337, 17.01816, 58.15749, 46.2..."
5,6,1,Hectochlorin M+H,667.115,CC(=O)O[C@@H]1c2nc(cs2)C(=O)O[C@@H](CCCC(C)(Cl...,C[C@H]1[C@@H](OC(C2=CSC([C@H](C(C)(OC(C3=CSC([...,NaN,106,"[200.205383, 201.312088, 202.184189, 205.25497...","[7905.953125, 2282.589844, 2641.422119, 581.43..."
6,7,1,Hectochlorin M+H,689.000,CC(=O)O[C@@H]1c2nc(cs2)C(=O)O[C@@H](CCCC(C)(Cl...,C[C@H]1[C@@H](OC(=O)C2=CSC(=N2)[C@H](C(OC(=O)C...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",55,"[264.268188, 336.234344, 366.046204, 387.89263...","[44.800793, 9.785302, 13.827823, 6.582838, 44...."
7,8,1,Cyclomarin A M+H-H2O,1025.610,CO[C@H](c1ccccc1)[C@@H]1NC(=O)[C@H](C)NC(=O)[C...,C[C@H]1C(=O)N[C@H](C(=O)N[C@H](C(=O)N([C@H](C(...,InChI=1S/C56H82N8O11/c1-30(2)24-34(8)44-52(70)...,500,"[100.112305, 101.115685, 102.116814, 103.05329...","[26031.130859, 2196.751953, 77.951561, 78.8012..."
8,9,1,Hectochlorin M+H,665.115,CC(=O)O[C@@H]1c2nc(cs2)C(=O)O[C@@H](CCCC(C)(Cl...,C[C@H]1[C@H](CCCC(C)(Cl)Cl)OC(=O)C2=CSC(=N2)[C...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",80,"[190.095718, 200.200897, 207.135315, 218.15692...","[122.539017, 3540.443359, 417.973267, 4853.495..."
9,10,1,Cyclomarin A M+Na,1065.600,CO[C@H](c1ccccc1)[C@@H]1NC(=O)[C@H](C)NC(=O)[C...,C[C@H]1C(=O)N[C@H](C(=O)N[C@H](C(=O)N([C@H](C(...,InChI=1S/C56H82N8O11/c1-30(2)24-34(8)44-52(70)...,52,"[485.273743, 486.284851, 532.348328, 534.77502...","[135.476303, 63.29353, 39.300949, 48.140911, 3..."



Columns:
['spec_index', 'scan', 'compound_name', 'precursor_mz', 'smiles', 'raw_smiles', 'inchi', 'num_peaks', 'mz_json', 'intensity_json']

Null counts in sample:
spec_index        0
scan              0
compound_name     0
precursor_mz      0
smiles            1
raw_smiles        1
inchi             3
num_peaks         0
mz_json           0
intensity_json    0
dtype: int64

Sample dtypes:
spec_index          int64
scan                int64
compound_name      object
precursor_mz      float64
smiles             object
raw_smiles         object
inchi              object
num_peaks           int64
mz_json            object
intensity_json     object
dtype: object


In [7]:
log("Running parsed CSV diagnostics...")

row_count = 0
smiles_non_null = 0
total_peaks_positive = 0

for chunk in pd.read_csv(PARSED_CSV, chunksize=CHUNK_SIZE):
    row_count += len(chunk)
    smiles_non_null += chunk["smiles"].notna().sum()
    total_peaks_positive += (chunk["num_peaks"] > 0).sum()

print("Total rows:", row_count)
print("Rows with non-null smiles:", smiles_non_null)
print("Rows with num_peaks > 0:", total_peaks_positive)
print("Fraction with smiles:", smiles_non_null / row_count if row_count else 0)

[20:08:30] Running parsed CSV diagnostics...
Total rows: 591719
Rows with non-null smiles: 448820
Rows with num_peaks > 0: 590709
Fraction with smiles: 0.758501924055168


In [8]:
def build_unique_smiles_file():
    if os.path.exists(UNIQUE_SMILES_CSV):
        log(f"{UNIQUE_SMILES_CSV} already exists. Skipping.")
        return

    log("Building unique SMILES file...")
    unique_smiles = set()
    total_rows = 0

    for chunk in pd.read_csv(PARSED_CSV, usecols=["smiles"], chunksize=CHUNK_SIZE):
        total_rows += len(chunk)
        vals = chunk["smiles"].dropna().astype(str).str.strip()
        vals = vals[vals != ""]
        unique_smiles.update(vals.tolist())

        log(f"Scanned {total_rows:,} rows | unique smiles so far: {len(unique_smiles):,}")

    unique_df = pd.DataFrame({"smiles": sorted(unique_smiles)})
    unique_df.to_csv(UNIQUE_SMILES_CSV, index=False)
    log(f"Saved unique SMILES -> {UNIQUE_SMILES_CSV} | count: {len(unique_df):,}")

In [9]:
build_unique_smiles_file()

[20:09:07] unique_smiles_full.csv already exists. Skipping.


In [10]:
unique_df = pd.read_csv(UNIQUE_SMILES_CSV)
print("Unique smiles shape:", unique_df.shape)
display(unique_df.head(10))
print("Any null smiles?", unique_df["smiles"].isna().any())
print("Any duplicated smiles?", unique_df["smiles"].duplicated().any())

Unique smiles shape: (30666, 1)


,smiles
0,Br.C=CC1CN2CCC1CC2C(O)c1ccnc2ccc(OC)cc12
1,Br.C=CCN1CCc2c(cc(O)c(O)c2Br)[C@@H](c2ccccc2)C1
2,Br.C=CCN1CCc2c(cc(O)c(O)c2Cl)C(c2ccccc2)C1
3,Br.C=CCN1CCc2cc(O)c(O)cc2C(c2ccccc2)C1
4,Br.CC(Cc1ccc(O)cc1)NCC(O)c1cc(O)cc(O)c1
5,Br.CC1NCCc2cc(O)c(O)cc21
6,Br.CCCN(CCC)C1CCc2ccc(O)cc2C1
7,Br.CCCN(CCC)C1CCc2cccc(O)c2C1
8,Br.CCCN1CC[C@@H]2c3cc(O)ccc3CC[C@@H]21
9,Br.CCCN1CCc2cc(O)cc3c2[C@H]1Cc1ccc(O)c(O)c1-3.O


Any null smiles? False
Any duplicated smiles? False


In [12]:
def load_label_mappings(index_json_path):
    with open(index_json_path, "r", encoding="utf-8") as f:
        index_data = json.load(f)

    pathway_mapping = {int(v): k for k, v in index_data["Pathway"].items()}
    superclass_mapping = {int(v): k for k, v in index_data["Superclass"].items()}
    class_mapping = {int(v): k for k, v in index_data["Class"].items()}

    return pathway_mapping, superclass_mapping, class_mapping

In [13]:
pathway_mapping, superclass_mapping, class_mapping = load_label_mappings(INDEX_JSON_PATH)

print("Number of pathway labels:", len(pathway_mapping))
print("Number of superclass labels:", len(superclass_mapping))
print("Number of class labels:", len(class_mapping))

print("\nPathway sample:", list(pathway_mapping.items())[:5])
print("Superclass sample:", list(superclass_mapping.items())[:5])
print("Class sample:", list(class_mapping.items())[:5])

Number of pathway labels: 7
Number of superclass labels: 77
Number of class labels: 687

Pathway sample: [(0, 'Alkaloids'), (1, 'Amino acids and Peptides'), (2, 'Carbohydrates'), (3, 'Fatty acids'), (4, 'Polyketides')]
Superclass sample: [(0, 'Alkylresorcinols'), (1, 'Amino acid glycosides'), (2, 'Aminosugars and aminoglycosides'), (3, 'Anthranilic acid alkaloids'), (4, 'Apocarotenoids')]
Class sample: [(0, '12-oxophytodienoic acid metabolites'), (1, '2-arylbenzofurans'), (2, '2-pyrone derivatives'), (3, '3-Decalinoyltetramic acids'), (4, '3-Spirotetramic acids')]


In [14]:
def smiles_to_fp(smiles, n_bits):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.tolist()

def call_model(url, fp2048, fp4096):
    payload = {
        "instances": [
            {
                "input_2048": fp2048,
                "input_4096": fp4096
            }
        ]
    }

    response = requests.post(url, json=payload, timeout=30)

    if response.status_code != 200:
        raise Exception(f"Server error: {response.status_code} | {response.text[:500]}")

    return response.json()

def get_top_k_indices(pred_json, k=3):
    scores = np.array(pred_json["predictions"][0])
    top_indices = scores.argsort()[-k:][::-1]
    return top_indices.tolist(), scores[top_indices].tolist()

def classify_smiles(smiles, pathway_mapping, superclass_mapping, class_mapping, top_k=3):
    if smiles is None or str(smiles).strip() == "":
        return None

    fp2048 = smiles_to_fp(smiles, 2048)
    fp4096 = smiles_to_fp(smiles, 4096)

    if fp2048 is None or fp4096 is None:
        return None

    class_pred = call_model(CLASS_URL, fp2048, fp4096)
    superclass_pred = call_model(SUPERCLASS_URL, fp2048, fp4096)
    pathway_pred = call_model(PATHWAY_URL, fp2048, fp4096)

    class_idx, class_scores = get_top_k_indices(class_pred, k=top_k)
    superclass_idx, superclass_scores = get_top_k_indices(superclass_pred, k=top_k)
    pathway_idx, pathway_scores = get_top_k_indices(pathway_pred, k=top_k)

    return {
        "class_indices": class_idx,
        "superclass_indices": superclass_idx,
        "pathway_indices": pathway_idx,
        "class_results": [class_mapping.get(i) for i in class_idx],
        "superclass_results": [superclass_mapping.get(i) for i in superclass_idx],
        "pathway_results": [pathway_mapping.get(i) for i in pathway_idx],
        "class_scores": class_scores,
        "superclass_scores": superclass_scores,
        "pathway_scores": pathway_scores
    }

def parse_label_result(result):
    if result is None:
        return {
            "top_class": None,
            "top_superclass": None,
            "top_pathway": None,
            "class_results_json": None,
            "superclass_results_json": None,
            "pathway_results_json": None,
            "class_scores_json": None,
            "superclass_scores_json": None,
            "pathway_scores_json": None
        }

    class_results = result.get("class_results", [])
    superclass_results = result.get("superclass_results", [])
    pathway_results = result.get("pathway_results", [])

    class_scores = result.get("class_scores", [])
    superclass_scores = result.get("superclass_scores", [])
    pathway_scores = result.get("pathway_scores", [])

    return {
        "top_class": class_results[0] if class_results else None,
        "top_superclass": superclass_results[0] if superclass_results else None,
        "top_pathway": pathway_results[0] if pathway_results else None,
        "class_results_json": json.dumps(class_results),
        "superclass_results_json": json.dumps(superclass_results),
        "pathway_results_json": json.dumps(pathway_results),
        "class_scores_json": json.dumps(class_scores),
        "superclass_scores_json": json.dumps(superclass_scores),
        "pathway_scores_json": json.dumps(pathway_scores)
    }

In [15]:
test_smiles_df = pd.read_csv(UNIQUE_SMILES_CSV, nrows=5)
test_smiles_list = test_smiles_df["smiles"].tolist()

test_rows = []
for smi in test_smiles_list:
    print("\nTesting:", smi)
    result = classify_smiles(
        smi,
        pathway_mapping=pathway_mapping,
        superclass_mapping=superclass_mapping,
        class_mapping=class_mapping,
        top_k=3
    )
    print(result)
    test_rows.append(result)

print("\nDone test predictions.")


Testing: Br.C=CC1CN2CCC1CC2C(O)c1ccnc2ccc(OC)cc12
{'class_indices': [423, 604, 518], 'superclass_indices': [63, 20, 57], 'pathway_indices': [0, 1, 5], 'class_results': ['Monocyclic monoterpenoids', 'Simple diketopiperazine alkaloids', 'Polyamines'], 'superclass_results': ['Small peptides', 'Fatty acyls', 'Pseudoalkaloids'], 'pathway_results': ['Alkaloids', 'Amino acids and Peptides', 'Shikimates and Phenylpropanoids'], 'class_scores': [0.135557294, 0.0456556976, 0.0188106596], 'superclass_scores': [0.0271695852, 0.0107278824, 0.00349417329], 'pathway_scores': [0.00955376, 0.00260898471, 0.00208172202]}

Testing: Br.C=CCN1CCc2c(cc(O)c(O)c2Br)[C@@H](c2ccccc2)C1
{'class_indices': [342, 367, 348], 'superclass_indices': [20, 57, 5], 'pathway_indices': [0, 4, 6], 'class_results': ['Iridoids monoterpenoids', 'Lactones', 'Isocoumarins'], 'superclass_results': ['Fatty acyls', 'Pseudoalkaloids', 'Aromatic polyketides'], 'pathway_results': ['Alkaloids', 'Polyketides', 'Terpenoids'], 'class_score

In [16]:
def classify_all_unique_smiles():
    unique_df = pd.read_csv(UNIQUE_SMILES_CSV)
    unique_df["smiles"] = unique_df["smiles"].astype(str)

    cache = load_cache(CACHE_JSON)

    already_done = set()
    if os.path.exists(SMILES_LABELS_CSV):
        done_df = pd.read_csv(SMILES_LABELS_CSV, usecols=["smiles"])
        already_done = set(done_df["smiles"].dropna().astype(str).tolist())
        log(f"Found existing labeled file with {len(already_done):,} rows")

    rows_to_save = []
    total = len(unique_df)

    log("Starting unique SMILES labeling...")

    for i, smiles in enumerate(unique_df["smiles"], start=1):
        if i % LOG_EVERY == 0:
            log(f"Processed {i:,}/{total:,} unique SMILES")

        if smiles in already_done:
            continue

        if smiles in cache:
            result = cache[smiles]
        else:
            try:
                result = classify_smiles(
                    smiles,
                    pathway_mapping=pathway_mapping,
                    superclass_mapping=superclass_mapping,
                    class_mapping=class_mapping,
                    top_k=3
                )
                cache[smiles] = result
            except Exception as e:
                result = {
                    "class_results": [],
                    "superclass_results": [],
                    "pathway_results": [],
                    "class_scores": [],
                    "superclass_scores": [],
                    "pathway_scores": [],
                    "error": str(e)
                }
                cache[smiles] = result

        parsed = parse_label_result(result)

        rows_to_save.append({
            "smiles": smiles,
            "top_class": parsed["top_class"],
            "top_superclass": parsed["top_superclass"],
            "top_pathway": parsed["top_pathway"],
            "class_results_json": parsed["class_results_json"],
            "superclass_results_json": parsed["superclass_results_json"],
            "pathway_results_json": parsed["pathway_results_json"],
            "class_scores_json": parsed["class_scores_json"],
            "superclass_scores_json": parsed["superclass_scores_json"],
            "pathway_scores_json": parsed["pathway_scores_json"],
            "label_status": "error" if (result is not None and "error" in result) else (
                "empty_label" if parsed["top_class"] is None and parsed["top_superclass"] is None and parsed["top_pathway"] is None else "ok"
            ),
            "label_error": result.get("error") if isinstance(result, dict) else None
        })

        if len(rows_to_save) >= LABEL_SAVE_EVERY:
            out_df = pd.DataFrame(rows_to_save)
            write_header = not os.path.exists(SMILES_LABELS_CSV)
            out_df.to_csv(SMILES_LABELS_CSV, mode="a", header=write_header, index=False)
            save_cache(cache, CACHE_JSON)
            log(f"Saved chunk -> {SMILES_LABELS_CSV}")
            rows_to_save = []

    if rows_to_save:
        out_df = pd.DataFrame(rows_to_save)
        write_header = not os.path.exists(SMILES_LABELS_CSV)
        out_df.to_csv(SMILES_LABELS_CSV, mode="a", header=write_header, index=False)

    save_cache(cache, CACHE_JSON)
    log("Finished unique SMILES labeling.")

In [17]:
classify_all_unique_smiles()

[20:19:34] Loaded cache with 30,666 entries
[20:19:35] Found existing labeled file with 30,666 rows
[20:19:35] Starting unique SMILES labeling...
[20:19:35] Processed 1,000/30,666 unique SMILES
[20:19:35] Processed 2,000/30,666 unique SMILES
[20:19:35] Processed 3,000/30,666 unique SMILES
[20:19:35] Processed 4,000/30,666 unique SMILES
[20:19:35] Processed 5,000/30,666 unique SMILES
[20:19:35] Processed 6,000/30,666 unique SMILES
[20:19:35] Processed 7,000/30,666 unique SMILES
[20:19:35] Processed 8,000/30,666 unique SMILES
[20:19:35] Processed 9,000/30,666 unique SMILES
[20:19:35] Processed 10,000/30,666 unique SMILES
[20:19:35] Processed 11,000/30,666 unique SMILES
[20:19:35] Processed 12,000/30,666 unique SMILES
[20:19:35] Processed 13,000/30,666 unique SMILES
[20:19:35] Processed 14,000/30,666 unique SMILES
[20:19:35] Processed 15,000/30,666 unique SMILES
[20:19:35] Processed 16,000/30,666 unique SMILES
[20:19:35] Processed 17,000/30,666 unique SMILES
[20:19:35] Processed 18,000/30

In [19]:
def merge_labels_back_to_spectra():
    if os.path.exists(LABELED_CSV):
        log(f"{LABELED_CSV} already exists. Delete it first if you want to rebuild.")
        return

    log("Loading smiles-label table...")
    labels_df = pd.read_csv(SMILES_LABELS_CSV)

    label_cols = [
        "smiles",
        "top_class",
        "top_superclass",
        "top_pathway",
        "class_results_json",
        "superclass_results_json",
        "pathway_results_json",
        "class_scores_json",
        "superclass_scores_json",
        "pathway_scores_json",
        "label_status",
        "label_error"
    ]
    labels_df = labels_df[label_cols].drop_duplicates(subset=["smiles"]).copy()

    # IMPORTANT: force smiles to string on label side
    labels_df["smiles"] = labels_df["smiles"].fillna("").astype(str).str.strip()

    log("Merging labels back to full spectra CSV in chunks...")
    total_rows = 0

    for chunk in pd.read_csv(PARSED_CSV, chunksize=CHUNK_SIZE):
        # IMPORTANT: force smiles to string on parsed side
        chunk["smiles"] = chunk["smiles"].fillna("").astype(str).str.strip()

        merged = chunk.merge(labels_df, on="smiles", how="left")

        # rows with no smiles / no structure
        no_structure_mask = merged["smiles"] == ""
        merged.loc[no_structure_mask, "label_status"] = "no_structure"

        write_header = not os.path.exists(LABELED_CSV)
        merged.to_csv(LABELED_CSV, mode="a", header=write_header, index=False)

        total_rows += len(chunk)
        log(f"Merged and saved {total_rows:,} rows -> {LABELED_CSV}")

    log("Finished final merge.")

In [20]:
merge_labels_back_to_spectra()

[20:21:14] Loading smiles-label table...
[20:21:14] Merging labels back to full spectra CSV in chunks...
[20:21:33] Merged and saved 50,000 rows -> spectra_with_labels_full.csv
[20:21:38] Merged and saved 100,000 rows -> spectra_with_labels_full.csv
[20:21:45] Merged and saved 150,000 rows -> spectra_with_labels_full.csv
[20:21:53] Merged and saved 200,000 rows -> spectra_with_labels_full.csv
[20:21:56] Merged and saved 250,000 rows -> spectra_with_labels_full.csv
[20:21:58] Merged and saved 300,000 rows -> spectra_with_labels_full.csv
[20:21:59] Merged and saved 350,000 rows -> spectra_with_labels_full.csv
[20:22:03] Merged and saved 400,000 rows -> spectra_with_labels_full.csv
[20:22:05] Merged and saved 450,000 rows -> spectra_with_labels_full.csv
[20:22:07] Merged and saved 500,000 rows -> spectra_with_labels_full.csv
[20:22:16] Merged and saved 550,000 rows -> spectra_with_labels_full.csv
[20:22:33] Merged and saved 591,719 rows -> spectra_with_labels_full.csv
[20:22:33] Finished 

In [21]:
final_df = pd.read_csv(LABELED_CSV, nrows=20)
print("Final sample shape:", final_df.shape)
display(final_df)

print("\nColumns:")
print(final_df.columns.tolist())

Final sample shape: (20, 21)


,spec_index,scan,compound_name,precursor_mz,smiles,raw_smiles,inchi,num_peaks,mz_json,intensity_json,...,top_superclass,top_pathway,class_results_json,superclass_results_json,pathway_results_json,class_scores_json,superclass_scores_json,pathway_scores_json,label_status,label_error
0,1,1,3-Des-Microcystein_LR M+H,981.540,C=C1C(=O)NC(C)C(=O)NC(CC(C)C)C(=O)NC(C(=O)O)C(...,CC(C)CC1NC(=O)C(C)NC(=O)C(=C)N(C)C(=O)CCC(NC(=...,NaN,218,"[289.286377, 295.545288, 298.489624, 317.32495...","[8068.0, 22507.0, 3925.0, 18742.0, 8604.0, 804...",...,Pseudoalkaloids,Alkaloids,"[""Polyamines"", ""m-Terphenyls"", ""Bergamotane se...","[""Pseudoalkaloids"", ""Histidine alkaloids"", ""Sm...","[""Alkaloids"", ""Fatty acids"", ""Carbohydrates""]","[0.00547501445, 0.00346270204, 0.00153490901]","[0.266518176, 0.00548121333, 0.00462335348]","[0.999282658, 0.000377744436, 0.000149309635]",ok,NaN
1,2,1,Hoiamide B M+H,940.250,CCC[C@@H](C)[C@H](O)[C@H](C)[C@H]1OC(=O)[C@H](...,CCC[C@@H](C)[C@@H]([C@H](C)[C@@H]1[C@H]([C@H](...,InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...,335,"[278.049927, 278.957642, 281.258667, 291.99609...","[35793.0, 47593.0, 95495.0, 115278.0, 91752.0,...",...,Pseudoalkaloids,Carbohydrates,"[""Flavonols"", ""Allohimachalane sesquiterpenoid...","[""Pseudoalkaloids"", ""Fatty acyls"", ""Meroterpen...","[""Carbohydrates"", ""Polyketides"", ""Alkaloids""]","[0.000483691692, 0.000349670649, 0.000222325325]","[0.126716733, 0.0578638315, 0.00303980708]","[0.0236490369, 0.0198290944, 0.00203564763]",ok,NaN
2,3,1,Malyngamide C M+H,456.100,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)[C@]12O[...,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)/[C@@]12...,InChI=1S/C24H38ClNO5/c1-3-4-5-6-8-11-19(30-2)1...,157,"[128.838745, 132.075684, 132.830322, 134.03843...","[8064.0, 8080.0, 18139.0, 11099.0, 4449.0, 256...",...,Pseudoalkaloids,Shikimates and Phenylpropanoids,"[""Pyrrole alkaloids"", ""Cycloeudesmane sesquite...","[""Pseudoalkaloids"", ""Isoflavonoids"", ""Phlorogl...","[""Shikimates and Phenylpropanoids"", ""Alkaloids...","[0.000493317842, 0.000487923622, 0.000382870436]","[0.14179635, 0.0231964, 0.00195267797]","[0.168069094, 0.0128429532, 0.005076617]",ok,NaN
3,4,1,Scytonemin M+H,545.000,O=C1C(C2=C3C(=Nc4ccccc43)/C(=C\c3ccc(O)cc3)C2=...,OC1=CC=C(\C=C2\C(=O)C(C3=C4C5=C(C=CC=C5)N=C4\C...,InChI=1S/C36H20N2O4/c39-21-13-9-19(10-14-21)17...,40,"[343.896484, 345.458496, 372.684021, 386.09747...","[142503.0, 67939.0, 175247.0, 65864.0, 104575....",...,Fatty acyls,Alkaloids,"[""Agarofuran sesquiterpenoids"", ""Purine alkalo...","[""Fatty acyls"", ""Docosanoids"", ""Cyclic polyket...","[""Alkaloids"", ""Terpenoids"", ""Polyketides""]","[0.000115284056, 0.000113921073, 8.32832739e-05]","[0.00116199255, 0.000619202852, 0.000615716]","[0.679835618, 0.00982105732, 0.00122019649]",ok,NaN
4,5,1,Salinisporamide A M+H,314.116,NaN,NaN,NaN,212,"[101.015465, 101.059807, 103.002441, 103.02854...","[34.201859, 14.90337, 17.01816, 58.15749, 46.2...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_structure,NaN
5,6,1,Hectochlorin M+H,667.115,CC(=O)O[C@@H]1c2nc(cs2)C(=O)O[C@@H](CCCC(C)(Cl...,C[C@H]1[C@@H](OC(C2=CSC([C@H](C(C)(OC(C3=CSC([...,NaN,106,"[200.205383, 201.312088, 202.184189, 205.25497...","[7905.953125, 2282.589844, 2641.422119, 581.43...",...,Saccharides,Carbohydrates,"[""Furocoumarins"", ""m-Terphenyls"", ""Monocyclic ...","[""Saccharides"", ""Phenolic acids (C6-C1)"", ""Ses...","[""Carbohydrates"", ""Alkaloids"", ""Fatty acids""]","[0.00144210458, 0.00118011236, 0.000242769718]","[0.00398671627, 0.00376001, 0.00178939104]","[0.0186070502, 0.0186050832, 0.00322005153]",ok,NaN
6,7,1,Hectochlorin M+H,689.000,CC(=O)O[C@@H]1c2nc(cs2)C(=O)O[C@@H](CCCC(C)(Cl...,C[C@H]1[C@@H](OC(=O)C2=CSC(=N2)[C@H](C(OC(=O)C...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",55,"[264.268188, 336.234344, 366.046204, 387.89263...","[44.800793, 9.785302, 13.827823, 6.582838, 44....",...,Peptide alkaloids,Alkaloids,"[""Furocoumarins"", ""m-Terphenyls"", ""Aporphine a...","[""Peptide alkaloids"", ""Ses


Columns:
['spec_index', 'scan', 'compound_name', 'precursor_mz', 'smiles', 'raw_smiles', 'inchi', 'num_peaks', 'mz_json', 'intensity_json', 'top_class', 'top_superclass', 'top_pathway', 'class_results_json', 'superclass_results_json', 'pathway_results_json', 'class_scores_json', 'superclass_scores_json', 'pathway_scores_json', 'label_status', 'label_error']


In [22]:
status_counts = {}

total_rows = 0
for chunk in pd.read_csv(LABELED_CSV, usecols=["label_status"], chunksize=CHUNK_SIZE):
    total_rows += len(chunk)
    vc = chunk["label_status"].value_counts(dropna=False).to_dict()
    for k, v in vc.items():
        status_counts[k] = status_counts.get(k, 0) + v

print("Total rows in final labeled CSV:", total_rows)
print("Final label_status counts:")
print(status_counts)

Total rows in final labeled CSV: 591719
Final label_status counts:
{'ok': 448820, 'no_structure': 142899}
